In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/imran495/dairy-farm-operations-and-financial-ledger-20192025/Farm 2019 Expenses vs. Income july to december.xlsx
/kaggle/input/datasets/imran495/dairy-farm-operations-and-financial-ledger-20192025/Farm Expenses Jan- June 2021.xlsx
/kaggle/input/datasets/imran495/dairy-farm-operations-and-financial-ledger-20192025/Desi Farm Report July to Dec 2023.xlsx
/kaggle/input/datasets/imran495/dairy-farm-operations-and-financial-ledger-20192025/Farm Expenses July-Dec 2020.xlsx
/kaggle/input/datasets/imran495/dairy-farm-operations-and-financial-ledger-20192025/Desi Farm Report July-Dec 2024.xlsx
/kaggle/input/datasets/imran495/dairy-farm-operations-and-financial-ledger-20192025/Desi Farm Report Jan-Jun 2023.xlsx
/kaggle/input/datasets/imran495/dairy-farm-operations-and-financial-ledger-20192025/Desi_Farm_Master_Data.csv
/kaggle/input/datasets/imran495/dairy-farm-operations-and-financial-ledger-20192025/Farm Expenses Jul-Dec 2022.xlsx
/kaggle/input/datasets/imran495/dairy-far

# Data Cleaning & ETL Pipeline: Desi Dairy Farm (2019–2025)

## Project Overview
This Data Engineering and ETL (Extract, Transform, Load) notebook demonstrates how to process raw, unstructured business ledgers into clean, analysis-ready datasets using Python (Pandas). 

## Business Context
The data originates from the operational ledgers of my own dairy business, Desi Farm (SMC-Private) Limited. I personally managed the farm and manually recorded all financial entries over a continuous 7-year period (2019–2025). This notebook processes these authentic, real-world logs to prepare them for financial analysis.

## The Data Challenge
The original data consists of 13 raw Excel files containing both income and expense records. These files present several real-world data wrangling challenges:
* **Differing Structural Frequencies:** Expenses were recorded regularly throughout the month (alongside aggregated end-of-month entries like salaries and "derra" expenses), whereas income was logged strictly on a monthly basis. This results in distinct row counts and structures that must be separated.
* **Unstructured Layouts:** The spreadsheets contain floating charts, merged cells, and irregular headers that need to be bypassed programmatically.
* **Multilingual Text:** The description logs contain a natural mix of English, Roman Urdu, and standard Urdu script.

## ETL Workflow
1. **Extract:** Read the 13 messy Excel files programmatically, targeting only the relevant sheets and data ranges.
2. **Transform:** Standardize column names, filter out empty rows, and split the data into two distinct DataFrames to handle the structural differences between income and expenses.
3. **Load:** Export the transformed data into two clean, separate CSV files (`Desi_Farm_Expenses.csv` and `Desi_Farm_Income.csv`) ready for financial forecasting and NLP categorization.

In [2]:
import pandas as pd

# 1. Define the exact path to your dataset
path = '/kaggle/input/datasets/imran495/dairy-farm-operations-and-financial-ledger-20192025/'

# 2. List of all 13 raw Excel files
file_names = [
    'Farm 2019 Expenses vs. Income july to december.xlsx',
    'Farm Expenses Jan-Jun 2020.xlsx',
    'Farm Expenses July-Dec 2020.xlsx',
    'Farm Expenses Jan- June 2021.xlsx',
    'Farm Expenses July-Dec 2021.xlsx',
    'Farm Expenses Jan-Jun 2022.xlsx',
    'Farm Expenses Jul-Dec 2022.xlsx',
    'Desi Farm Report Jan-Jun 2023.xlsx',
    'Desi Farm Report July to Dec 2023.xlsx',
    'Desi Farm Report Jan-June 2024.xlsx',
    'Desi Farm Report July-Dec 2024.xlsx',
    'Desi Farm Report Jan-June 2025.xlsx',
    'Desi Farm Report July to Dec 2025.xlsx'
]

expenses_list = []
income_list = []

# 3. Loop through files to extract the data
for name in file_names:
    full_path = path + name
    
    # Read the 'Transactions' sheet
    df = pd.read_excel(full_path, sheet_name='Transactions')
    
    # --- EXTRACT EXPENSES ---
    # Select from row 3 onwards (to skip extra headers) and columns 1, 2, 3, 4
    df_exp = df.iloc[3:, [1, 2, 3, 4]].copy() 
    expenses_list.append(df_exp)
    
    # --- EXTRACT INCOME ---
    # Select from row 3 onwards (to skip extra headers) and columns 6, 7, 8, 9
    df_inc = df.iloc[3:, [6, 7, 8, 9]].copy() 
    income_list.append(df_inc)

# 4. Combine all individual dataframes into two master dataframes
final_expenses = pd.concat(expenses_list, ignore_index=True)
final_income = pd.concat(income_list, ignore_index=True)

# 5. Standardize column names for both datasets
columns_setup = ['date', 'amount', 'description', 'category']
final_expenses.columns = columns_setup
final_income.columns = columns_setup

# 6. Drop completely empty rows
final_expenses = final_expenses.dropna(subset=['date', 'amount'], how='all')
final_income = final_income.dropna(subset=['date', 'amount'], how='all')

# 7. FIX ERROR: Convert 'date' column to proper Datetime format FIRST
final_expenses['date'] = pd.to_datetime(final_expenses['date'])
final_income['date'] = pd.to_datetime(final_income['date'])

# 8. DATA INTEGRITY FILTER: Remove incomplete data from November 2025 onwards
# We remove this because the missing income would skew the profit/loss analysis
final_expenses = final_expenses[final_expenses['date'] < '2025-11-01'] 
final_income = final_income[final_income['date'] < '2025-11-01'] 

# 9. Export the cleaned data to two separate CSV files
final_expenses.to_csv('/kaggle/working/Desi_Farm_Expenses.csv', index=False)
final_income.to_csv('/kaggle/working/Desi_Farm_Income.csv', index=False)

print("✅ Success! Data aligned, dates converted, and incomplete months filtered out.")

# Preview the clean data
print("\n--- EXPENSES DATA PREVIEW ---")
display(final_expenses.head())

print("\n--- INCOME DATA PREVIEW ---")
display(final_income.head())

✅ Success! Data aligned, dates converted, and incomplete months filtered out.

--- EXPENSES DATA PREVIEW ---


,date,amount,description,category
12,2019-07-03,3300,پیاز 75 کلو,Feed
13,2019-07-06,2400,10 کلو تیل، 1کلو درپھڑ مرچ,Feed
14,2019-07-13,23750,"10 توڑے ونڈا , 5 توڑے کھل",Feed
15,2019-07-31,1800,15کلوگرام جوار بیج,Seeds
16,2019-07-31,1500,10 مورکاں,Animal Husbandry



--- INCOME DATA PREVIEW ---


,date,amount,description,category
0,2019-12-31,360000,25lites x 30 x 4 x 80 price,Milk @Rs80
1,2019-12-31,84000,1200 ltr. in December,Milk @Rs70
64,2020-01-05,1000,ٹیوب ویل سلطان,NaN
65,2020-01-10,24430,نوید 349 لیٹر,NaN
66,2020-01-15,1120,14 لیٹر فوجی اسامہ,NaN



> Data Integrity Note: Handling Incomplete Records (Nov-Dec 2025):  While reviewing the extracted data for late 2025, a structural imbalance was identified: the daily expenses for November 2025 were fully logged, but the corresponding monthly income records were missing or highly incomplete. To maintain the integrity of the financial analysis and prevent a skewed profit/loss calculation (which would artificially show a massive operational loss for that month), all transaction data from November 1, 2025, onwards has been explicitly excluded from the final dataset. This step ensures the upcoming case study will reflect accurate, balanced, and reliable operational margins.


## ✅ ETL Pipeline Complete

### Summary of Achievements
We have successfully transformed the raw financial logs of Desi Farm (SMC-Private) Limited. By programmatically iterating through 13 messy Excel files, we accomplished the following:
* **Structural Separation:** Successfully split the data into two distinct datasets (`Desi_Farm_Expenses.csv` and `Desi_Farm_Income.csv`) to account for their different recording frequencies.
* **Standardization:** Applied uniform, lowercase column headers (`date`, `amount`, `description`, `category`) across all years of data.
* **Data Cleaning:** Stripped out empty rows, formatting artifacts, and irrelevant spreadsheet noise.

### The Result
The output consists of two highly structured, machine-readable CSV files ready for immediate analysis. 

### Next Steps for Analysis
With the data now cleaned and isolated, future projects can focus on:
1. **Time-Series Forecasting:** Analyzing seasonal trends in feed prices and electricity costs.
2. **Profitability Margins:** Merging the income and expense datasets by month to calculate net operational margins.
3. **NLP Categorization:** Using Natural Language Processing to extract specific vendor names and items from the multilingual `description` column.